# Tutorial: Deterministic Lot-Sizing — Three Formulations

This notebook implements and compares three mathematical formulations of the 
deterministic production planning problem.

## Demand Structure

**Two demand states:**
- **Normal demand:** Regular production scenario (d_N)
- **Disaster/Bursty demand:** High demand spike scenario (d_D)

Each formulation is tested against both demand scenarios to show robustness.

## Three Formulations Compared

1. **Equality balance constraint** - Strict inventory balance
2. **Inequality balance constraint** - Separated inventory/backlog balance
3. **Convex backlog cost approximation** - Piecewise-linear cost modeling

--

In [ ]:
# Imports

import random
import gurobipy as gp
from gurobipy import GRB

random.seed(42)

In [200]:
# Sets and Parameters

N = 12  # The length of the planning horizon
C = random.randint(150, 250)  # Production capacity

# Define normal and bursty demands dynamically based on capacity and alpha
alpha = random.randint(7, 10)/10  # Threshold parameter for classifying demands
dn = [random.randint(1, int(alpha * C)) for t in range(N)]  # Normal demand
db = [random.choice([0, random.randint(int(alpha * C) + 1, C * 2)]) for t in range(N)]  # Bursty demand

# Having just normal demand
#dn = [random.randint(1, 140) for t in range(N)]  # Random normal demand
#db = 0

All_demands = [dn[t] + db[t] for t in range(N)]
Total_demand = sum(All_demands)

random_value_1 = random.randint(15, 40)
random_value_2 = random.randint(30, 60)
random_value_3 = random.randint(5, 15)
random_value_4 = random.randint(60, 80)

p = [random_value_1 for _ in range(N)]  # Random production cost
b = [random_value_2 for _ in range(N)]  # Random backlog cost
h = [random_value_3 for _ in range(N)]  # Random holding cost
q = [random_value_4 for _ in range(N)]  # Random setup cost

initial_inventory = 0
initial_backlog   = 0

In [201]:
# calculate the parameters of the piecewise linear approximation of the convex backlog cost.

def quadratic_backlog_cost(y, b):
    return b * y**2

def compute_piecewise_coefficients(b, breakpoints):
    
    a_j = []
    b_j = []

    # Iterate over the breakpoints to compute slopes and intercepts for each segment
    for j in range(len(breakpoints) - 1):
        y_j = breakpoints[j]
        y_j_next = breakpoints[j + 1]
        
        # Compute slope b_j
        slope = (quadratic_backlog_cost(y_j_next, b) - quadratic_backlog_cost(y_j, b)) / (y_j_next - y_j)
        b_j.append(slope)
        
        # Compute intercept a_j
        intercept = quadratic_backlog_cost(y_j, b) - slope * y_j
        a_j.append(intercept)

    return a_j, b_j

breakpoints = [0, 150, 300, 400, 500, 550, 600]  # Non-linear breakpoints to approximate the quadratic backlog cost.

a_j, b_j = compute_piecewise_coefficients(random_value_2, breakpoints)

In [ ]:
# Function for model with equality balance constraint:

def Model_with_equality_balance_constraint(dn, db, h, b, p, q, C, N, initial_inventory, initial_backlog):
    model = gp.Model("Model_with_equality_balance_constraint")

    # Decision variables
    x = model.addVars(N, vtype=GRB.CONTINUOUS, lb=0, name="Production")
    y = model.addVars(N, vtype=GRB.BINARY, name="Setup")
    s = model.addVars(N, vtype=GRB.CONTINUOUS, lb=0, name="Inventory")
    r = model.addVars(N, vtype=GRB.CONTINUOUS, lb=0, name="Backlog")


    # Objective function
    model.setObjective(
        gp.quicksum(h[t] * s[t] + b[t] * r[t] + p[t] * x[t] + q[t] * y[t] for t in range(N)),
        GRB.MINIMIZE
    )

    # Constraints
    for t in range(N):
        if t == 0:
            # Initial period constraints
            model.addConstr(initial_inventory - initial_backlog + x[t] == dn[t] + db[t] + s[t] - r[t], name=f"Balance_{t}")
        else:
            # Inventory balance constraints
            model.addConstr(s[t-1] - r[t-1] + x[t] == dn[t] + db[t] + s[t] - r[t], name=f"Balance_{t}")

        # Production capacity constraints
        model.addConstr(x[t] <= C * y[t], name=f"Capacity_{t}")
        
    return model

# Function for model with two inequality balance constraints:
def Model_with_two_inequality_balance_constraints(dn, db, h, b, p, q, C, N, initial_inventory, initial_backlog):
    model = gp.Model("Model_with_two_inequality_balance_constraints")

    # Decision variables
    x2 = model.addVars(N, vtype=GRB.CONTINUOUS, lb=0, name="Production")
    y2 = model.addVars(N, vtype=GRB.BINARY, name="Setup")
    H2 = model.addVars(N, vtype=GRB.CONTINUOUS, lb=0, name="Overall_holding_and_backorder_cost")

    # Objective function
    model.setObjective(
        gp.quicksum(p[t] * x2[t] + q[t] * y2[t] + H2[t] for t in range(N)),
        GRB.MINIMIZE
    )

    # Constraints
    for t in range(N):
        # Two inequality for balance 
        model.addConstr(H2[t] >= h[t] * (initial_inventory-initial_backlog + gp.quicksum(x2[i] - (dn[i] + db[i]) for i in range(t+1))), name=f"Balance_1_{t}")
        model.addConstr(H2[t] >= -b[t] * (initial_inventory-initial_backlog + gp.quicksum(initial_inventory-initial_backlog + x2[i] - (dn[i] + db[i]) for i in range(t+1))), name=f"Balance_2_{t}")

        # Production capacity constraints
        model.addConstr(x2[t] <= C * y2[t], name=f"Capacity_{t}")
        
    return model

# Function for model with two inequality balance constraints and convex backlog cost:
def Model_with_convex_backlog_cost(dn, db, h, p, q, C, N, initial_inventory, initial_backlog, breakpoints, a_j, b_j):
    model = gp.Model("Model_with_convex_backlog_cost")

    # Decision variables
    x3 = model.addVars(N, vtype=GRB.CONTINUOUS, lb=0, name="Production")
    y3 = model.addVars(N, vtype=GRB.BINARY, name="Setup")
    H3 = model.addVars(N, vtype=GRB.CONTINUOUS, lb=0, name="Overall_holding_and_backorder_cost")

    # Objective function
    model.setObjective(
        gp.quicksum(p[t] * x3[t] + q[t] * y3[t] + H3[t] for t in range(N)),
        GRB.MINIMIZE
    )

    # Constraints
    for t in range(N):
        # Subsequent periods
        model.addConstr(H3[t] >= h[t] * (initial_inventory-initial_backlog + gp.quicksum(x3[i] - (dn[i] + db[i]) for i in range(t+1))), name=f"Balance_1_{t}")
        # The approximation of the convex backlog cost introducing a set of linear segments
        for j in range(len(breakpoints) - 1):
            model.addConstr(H3[t] >= -b_j[j] * (initial_inventory-initial_backlog + gp.quicksum(x3[i] - (dn[i] + db[i]) for i in range(t+1))) + a_j[j], name=f"Balance_2_Backlog_Cost_Segment_{t}_{j}")

        # Production capacity constraints
        model.addConstr(x3[t] <= C * y3[t], name=f"Capacity_{t}")
    
    return model


# Run the model with equality balance constraint
model_1 = Model_with_equality_balance_constraint(dn, db, h, b, p, q, C, N, initial_inventory, initial_backlog)
model_1.setParam(GRB.Param.OutputFlag, 1)
model_1.optimize()

# Run the model with two inequality balance constraints
model_2 = Model_with_two_inequality_balance_constraints(dn, db, h, b, p, q, C, N, initial_inventory, initial_backlog)
model_2.setParam(GRB.Param.OutputFlag, 1)
model_2.optimize()

# Run the model with two inequality balance constraints and convex backlog cost
model_3 = Model_with_convex_backlog_cost(dn, db, h, p, q, C, N, initial_inventory, initial_backlog, breakpoints, a_j, b_j)
model_3.setParam(GRB.Param.OutputFlag, 1)
model_3.optimize()

Gurobi Optimizer version 11.0.0 build v11.0.0rc2 (win64 - Windows 10.0 (19045.2))

CPU model: 11th Gen Intel(R) Core(TM) i5-1145G7 @ 2.60GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 24 rows, 48 columns and 82 nonzeros
Model fingerprint: 0x60ea31b7
Variable types: 36 continuous, 12 integer (12 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+02]
  Objective range  [1e+01, 7e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+01, 4e+02]

CPU model: 11th Gen Intel(R) Core(TM) i5-1145G7 @ 2.60GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 24 rows, 48 columns and 82 nonzeros
Model fingerprint: 0x60ea31b7
Variable types: 36 continuous, 12 integer (12 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+02]
  Objective range  [1e+01, 7e+01]
  Bounds range     [1e+00, 1e+

In [203]:
# Data

print(" **** Parameters ****")
print()
print("Demands:")
print("      Normal (dn):", dn)
print("      Bursty (db):", db)
print("      All demands:", All_demands)
print("     Total demand:", Total_demand)
print()
print("Costs:")
print("   Production (p):", random_value_1)
print("      Holding (h):", random_value_3)
print("        Setup (q):", random_value_4)
print("      Backlog (b):", random_value_2)
print()
print("Initial Inventory:", initial_inventory)
print("  Initial Backlog:", initial_backlog)
print()
print("  Percentage of Capacity (C) to consider a demand as a bursty demand (alpha):", alpha)
print()
print(" **** parameters of piecewise linear approximation of convex backlog cost ****")
print(" intercept a_j:", a_j)
print("     slope b_j:", b_j)
print("   breakpoints:", breakpoints)

 **** Parameters ****

Demands:
      Normal (dn): [42, 14, 128, 162, 156, 95, 59, 4, 57, 120, 42, 128]
      Bursty (db): [173, 0, 0, 168, 0, 309, 172, 324, 303, 0, 0, 0]
      All demands: [215, 14, 128, 330, 156, 404, 231, 328, 360, 120, 42, 128]
     Total demand: 2456

Costs:
   Production (p): 39
      Holding (h): 10
        Setup (q): 67
      Backlog (b): 30

Initial Inventory: 0
  Initial Backlog: 0

  Percentage of Capacity (C) to consider a demand as a bursty demand (alpha): 1.0

 **** parameters of piecewise linear approximation of convex backlog cost ****
 intercept a_j: [0.0, -1350000.0, -3600000.0, -6000000.0, -8250000.0, -9900000.0]
     slope b_j: [4500.0, 13500.0, 21000.0, 27000.0, 31500.0, 34500.0]
   breakpoints: [0, 150, 300, 400, 500, 550, 600]


In [204]:
# Results for the model with equality balance constraint

print("**** Results of the Model with Equality Balance Constraint ****")
print()
if model_1.status == GRB.OPTIMAL:
    production_plan1 = [model_1.getVarByName(f"Production[{t}]").x for t in range(N)]
    All_production1 = sum(production_plan1)
    Setup1 = [model_1.getVarByName(f"Setup[{t}]").x for t in range(N)]
    inventory_plan1 = [model_1.getVarByName(f"Inventory[{t}]").x for t in range(N)]
    backlog_plan1 = [model_1.getVarByName(f"Backlog[{t}]").x for t in range(N)]
    Inventory_cost1 = sum(inventory_plan1[t] * h[t] for t in range(N))
    Backlog_cost1 = sum(backlog_plan1[t] * b[t] for t in range(N))
    
    print("      Objective Value:", model_1.objVal)
    print("     Total Production:", All_production1)
    print("      Production Plan:", production_plan1)
    print("           Setup Plan:", Setup1)
    print()
    print("       Inventory Plan:", inventory_plan1)
    print("         Backlog Plan:", backlog_plan1)
    print(" Total Inventory Cost:", Inventory_cost1)
    print("   Total Backlog Cost:", Backlog_cost1)
else:
    print("No optimal solution found for the model with equality balance constraint.")

**** Results of the Model with Equality Balance Constraint ****

      Objective Value: 184553.0
     Total Production: 1804.0
      Production Plan: [164.0, 164.0, 164.0, 164.0, 164.0, 164.0, 164.0, 164.0, 164.0, 164.0, 164.0, 0.0]
           Setup Plan: [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0]

       Inventory Plan: [0.0, 99.0, 135.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
         Backlog Plan: [51.0, 0.0, 0.0, 31.0, 23.0, 263.0, 330.0, 494.0, 690.0, 646.0, 524.0, 652.0]
 Total Inventory Cost: 2340.0
   Total Backlog Cost: 111120.0


In [205]:
# Results for the model with two inequality balance constraints

print("**** Results of the Model with Two Inequality Balance Constraints ****")
print()
if model_2.status == GRB.OPTIMAL:
    production_plan2 = [model_2.getVarByName(f"Production[{t}]").x for t in range(N)]
    All_production2 = sum(production_plan2)
    Setup2 = [model_2.getVarByName(f"Setup[{t}]").x for t in range(N)]
    Inventory_Backlog_plan2 = [model_2.getVarByName(f"Overall_holding_and_backorder_cost[{t}]").x for t in range(N)]
    Details2 = [production_plan2[t] - All_demands[t] for t in range(N)]

    print("                  Objective Value:", model_2.objVal)
    print("                 Total Production:", All_production2)
    print("                  Production Plan:", production_plan2)
    print("                       Setup Plan:", Setup2)
    print()
    print("    Inventory and Backlog Details:")
    print("     Production-Demand Difference:", Details2)
    print("         Holding or Backlog Costs:", Inventory_Backlog_plan2)
    
else:
    print("No optimal solution found for the model with two inequality constraints.")

**** Results of the Model with Two Inequality Balance Constraints ****

                  Objective Value: 184553.0
                 Total Production: 1804.0
                  Production Plan: [164.0, 164.0, 164.0, 164.0, 164.0, 164.0, 164.0, 164.0, 164.0, 164.0, 164.0, 0.0]
                       Setup Plan: [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0]

    Inventory and Backlog Details:
     Production-Demand Difference: [-51.0, 150.0, 36.0, -166.0, 8.0, -240.0, -67.0, -164.0, -196.0, 44.0, 122.0, -128.0]
         Holding or Backlog Costs: [1530.0, 990.0, 1350.0, 930.0, 690.0, 7890.0, 9900.0, 14820.0, 20700.0, 19380.0, 15720.0, 19560.0]


In [206]:
# Results for the model with two inequality balance constraints and convex backlog cost

print("**** Results of the Model with Two Inequality Balance Constraints and convex backlog cost ****")
print()
if model_3.status == GRB.OPTIMAL:
    production_plan3 = [model_3.getVarByName(f"Production[{t}]").x for t in range(N)]
    All_production3 = sum(production_plan3)
    Setup3 = [model_3.getVarByName(f"Setup[{t}]").x for t in range(N)]
    Inventory_Backlog_plan3 = [model_3.getVarByName(f"Overall_holding_and_backorder_cost[{t}]").x for t in range(N)]
    Details3 = [production_plan3[t] - All_demands[t] for t in range(N)]

    print("                  Objective Value:", model_3.objVal)
    print("                 Total Production:", All_production3)
    print("                  Production Plan:", production_plan3)
    print("                       Setup Plan:", Setup3)
    print()
    print("    Inventory and Backlog Details:")
    print("     Production-Demand Difference:", Details3)
    print("         Holding or Backlog Costs:", Inventory_Backlog_plan3)

else:
    print("No optimal solution found for the model with two inequality constraints.")

**** Results of the Model with Two Inequality Balance Constraints and convex backlog cost ****

                  Objective Value: 55144896.0
                 Total Production: 1968.0
                  Production Plan: [164.0, 164.0, 164.0, 164.0, 164.0, 164.0, 164.0, 164.0, 164.0, 164.0, 164.0, 164.0]
                       Setup Plan: [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]

    Inventory and Backlog Details:
     Production-Demand Difference: [-51.0, 150.0, 36.0, -166.0, 8.0, -240.0, -67.0, -164.0, -196.0, 44.0, 122.0, 36.0]
         Holding or Backlog Costs: [229500.0, 990.0, 1350.0, 139500.0, 103500.0, 2200500.0, 3330000.0, 7338000.0, 13905000.0, 12387000.0, 8256000.0, 7176000.0]


In [207]:
#Comparisons

print("**** Comparisons ****")
print()
print("      Objective Value 1:", model_1.objVal)
print("      Objective Value 2:", model_2.objVal)
print("      Objective Value 3:", model_3.objVal)
print("           Total Production 1:", All_production1)
print("           Total Production 2:", All_production2)
print("           Total Production 3:", All_production3)
print("      Production Plan 1:", production_plan1)
print("      Production Plan 2:", production_plan2)
print("      Production Plan 3:", production_plan3)

**** Comparisons ****

      Objective Value 1: 184553.0
      Objective Value 2: 184553.0
      Objective Value 3: 55144896.0
           Total Production 1: 1804.0
           Total Production 2: 1804.0
           Total Production 3: 1968.0
      Production Plan 1: [164.0, 164.0, 164.0, 164.0, 164.0, 164.0, 164.0, 164.0, 164.0, 164.0, 164.0, 0.0]
      Production Plan 2: [164.0, 164.0, 164.0, 164.0, 164.0, 164.0, 164.0, 164.0, 164.0, 164.0, 164.0, 0.0]
      Production Plan 3: [164.0, 164.0, 164.0, 164.0, 164.0, 164.0, 164.0, 164.0, 164.0, 164.0, 164.0, 164.0]
